# 第1章 导论 —— 配套代码

对应正文 `book/part1/01-introduction.md`，可逐格运行。

> 运行前请先生成内置数据：在项目根目录执行 `uv run python scripts/make_sample_data.py`。

本 notebook 帮助你：加载数据 → 计算收益 → 作图 → 看清金融数据的特征 → **亲手制造一次「前视偏差」**，体会它如何骗人。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

## 1. 环境与中文字体

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fds import (load_sample_prices, daily_returns, set_chinese_font,
                 annualized_return, annualized_volatility, list_datasets)

set_chinese_font()
print("可用内置数据集：", list_datasets())

## 2. 加载内置示例数据

四只虚构 A 股风格股票：`BANK`（银行，低波动）、`LIQUOR`（白酒，高成长高波动）、`TECH`（科技，最高波动）、`UTILITY`（公用事业，最稳）。

In [ ]:
prices = load_sample_prices()
print("数据形状：", prices.shape)
print("日期范围：", prices.index.min().date(), "~", prices.index.max().date())
prices.tail()

## 3. 简单收益率 vs 对数收益率

组合层面用简单收益率，建模用对数收益率；小幅变动时两者近似相等。

In [ ]:
simple = daily_returns(prices)
log_ret = daily_returns(prices, log=True)
compare = pd.DataFrame({"简单收益": simple["BANK"], "对数收益": log_ret["BANK"]})
compare.head()

## 4. 累计净值曲线

In [ ]:
cumulative = (1 + simple).cumprod()
ax = cumulative.plot(title="四只示例股票累计净值（初始=1）")
ax.set_ylabel("净值"); ax.set_xlabel("日期")
plt.tight_layout(); plt.show()

## 5. 基础统计：收益率的四个矩

注意每只股票的**超额峰度远大于 0**——这正是金融收益率「厚尾」的数字证据（详见第4章）。

In [ ]:
stats = pd.DataFrame({
    "年化收益": annualized_return(simple),
    "年化波动": annualized_volatility(simple),
    "日均收益": simple.mean(),
    "偏度": simple.skew(),
    "超额峰度": simple.kurtosis(),
})
stats.round(4)

## 6. 信噪比有多低？

把「日均收益」和「日波动」放在一起看，就能直观感受到正文 1.6.1 说的**低信噪比**：信号（均值）淹没在噪声（波动）里。

In [ ]:
ir = simple.mean() / simple.std()
print("日度信息比（均值/标准差）：")
print(ir.round(4))
print("\n解读：数值普遍很小（量级 0.0x），说明单日可预测性极弱。")

## 7. 收益率分布：厚尾的样子

In [ ]:
from scipy.stats import norm

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(simple["TECH"], bins=60, density=True, alpha=0.6, label="TECH 实际分布")
x = np.linspace(simple["TECH"].min(), simple["TECH"].max(), 200)
ax.plot(x, norm.pdf(x, simple["TECH"].mean(), simple["TECH"].std()), "r--", label="正态分布")
ax.set_title("TECH 日收益分布 vs 正态：注意两端的厚尾")
ax.set_xlabel("日收益率"); ax.legend()
plt.tight_layout(); plt.show()

## 8. 亲手制造一次「前视偏差」

正文强调：**用到未来信息会让回测虚高**。下面用一个夸张的例子演示。

任务：预测明天涨跌。
- **错误做法**：把「明天的收益符号」当特征——相当于偷看答案。
- **正确做法**：只用「今天及以前」的信息（动量假设）。

In [ ]:
r = simple["TECH"]
future_up = (r.shift(-1) > 0).astype(int)        # 明天是否上涨（标签）

# 错误：用『明天收益的符号』当特征（泄露未来！）
leaky_feature = (r.shift(-1) > 0).astype(int)
valid_mask = r.shift(-1).notna()
acc_leak = (leaky_feature[valid_mask] == future_up[valid_mask]).mean()

# 正确：用『今天收益的符号』预测明天（只用已知信息）
valid_feature = (r > 0).astype(int)
acc_valid = (valid_feature[valid_mask] == future_up[valid_mask]).mean()

print(f"含前视偏差的『准确率』：{acc_leak:.1%}  <- 看似神准，其实是作弊")
print(f"无偏差的真实准确率：    {acc_valid:.1%}  <- 接近抛硬币，才是现实")

> ⚠️ 第一个「100% 准确率」完全是假象——因为特征里混入了答案。真实金融预测准确率很难稳定超过 55%。**任何高得离谱的回测结果，先怀疑数据泄露。**

## 习题参考解答

**习题1：只画出波动率最高与最低的两只股票的累计净值。**

In [ ]:
vol = annualized_volatility(simple).sort_values()
low, high = vol.index[0], vol.index[-1]
print(f"波动最低：{low}（{vol[low]:.1%}），波动最高：{high}（{vol[high]:.1%}）")
ax = cumulative[[low, high]].plot(title=f"波动最低（{low}）vs 最高（{high}）")
ax.set_ylabel("净值"); plt.tight_layout(); plt.show()

**习题2：对数收益率累计是否与简单收益率一致？** `exp(对数收益累加)` 应与简单收益累乘净值高度一致。

In [ ]:
nav_simple = (1 + simple["BANK"]).cumprod()
nav_log = np.exp(log_ret["BANK"].cumsum())
print("两种方式最终净值：", round(nav_simple.iloc[-1], 6), round(nav_log.iloc[-1], 6))
print("是否高度一致：", np.allclose(nav_simple.values, nav_log.values))

**习题3（思考题）：为什么金融预测比图像识别更难？**

参考要点：(1) 低信噪比；(2) 非平稳，规律会失效；(3) 反身性——赚钱规律一旦被广泛使用就会被套利消失；(4) 厚尾黑天鹅难以从历史学到。